# Problem statement
 Understand forward propagation, cross-entropy loss, 3D Jacobians, and backpropagation without PyTorch or TensorFlow.

In [8]:
import numpy as np

# This makes the random values the same every time we run the notebook.
np.random.seed(42)

## 1. Create Simple Training Data

We create 100 samples with 4 input features and 3 possible classes.

In [9]:
# 100 samples and 4 input features
X = np.random.randn(100, 4)

# Class labels: 0, 1, or 2
y = np.random.randint(0, 3, 100)

# Convert the labels into one-hot form
Y = np.zeros((100, 3))
Y[np.arange(100), y] = 1

print("Input shape:", X.shape)
print("Label shape:", Y.shape)

Input shape: (100, 4)
Label shape: (100, 3)


## 2. Define the Neural Network

In [10]:
input_size = 4
hidden_size = 6
output_size = 3

learning_rate = 0.01

# Initialize weights with small random values
W1 = np.random.randn(input_size, hidden_size) * 0.1
b1 = np.zeros((1, hidden_size))

W2 = np.random.randn(hidden_size, output_size) * 0.1
b2 = np.zeros((1, output_size))

## 3. Activation Functions

In [11]:
def relu(x):
    # ReLU changes negative values to zero.
    return np.maximum(0, x)


def relu_derivative(x):
    # The gradient is 1 for positive values and 0 otherwise.
    return (x > 0).astype(float)


def softmax(x):
    # Subtracting the maximum helps avoid very large exponentials.
    x = x - np.max(x, axis=1, keepdims=True)

    exp_x = np.exp(x)

    return exp_x / np.sum(exp_x, axis=1, keepdims=True)

## 4. Training the Network

The loop below performs:

1. Forward propagation
2. Cross-entropy loss calculation
3. Softmax Jacobian calculation
4. Backpropagation
5. Weight updates

In [12]:
for epoch in range(100):

    # =========================
    # FORWARD PROPAGATION
    # =========================

    # First layer
    Z1 = X @ W1 + b1
    A1 = relu(Z1)

    # Output layer
    Z2 = A1 @ W2 + b2
    A2 = softmax(Z2)


    # =========================
    # CROSS-ENTROPY LOSS
    # =========================

    loss = -np.mean(
        np.sum(Y * np.log(A2 + 1e-8), axis=1)
    )


    # =========================
    # BACKPROPAGATION
    # =========================

    # ----- Softmax Jacobian -----

    # We create one Jacobian matrix for every sample.
    batch_size = X.shape[0]

    J_softmax = np.zeros(
        (batch_size, output_size, output_size)
    )

    for i in range(batch_size):

        p = A2[i]

        # Softmax Jacobian:
        # J = diag(p) - p.p^T
        J_softmax[i] = (
            np.diag(p) - np.outer(p, p)
        )


    # Gradient of the loss with respect to output probabilities
    dA2 = -Y / (A2 + 1e-8)

    # Apply the Jacobian to every sample in the batch
    dZ2 = np.einsum(
        'bij,bj->bi',
        J_softmax,
        dA2
    )

    dZ2 = dZ2 / batch_size


    # ----- Gradients for W2 and b2 -----

    # Calculate a weight gradient for every sample.
    # This gives us a 3D matrix.
    dW2_each = np.einsum(
        'bi,bj->bij',
        A1,
        dZ2
    )

    # Combine the gradients from all samples
    dW2 = np.sum(dW2_each, axis=0)

    db2 = np.sum(dZ2, axis=0, keepdims=True)


    # ----- Move the gradient to the hidden layer -----

    dA1 = dZ2 @ W2.T

    dZ1 = dA1 * relu_derivative(Z1)


    # ----- Gradients for W1 and b1 -----

    # Again, calculate the gradient separately for each sample.
    dW1_each = np.einsum(
        'bi,bj->bij',
        X,
        dZ1
    )

    dW1 = np.sum(dW1_each, axis=0)

    db1 = np.sum(dZ1, axis=0, keepdims=True)


    # =========================
    # UPDATE WEIGHTS
    # =========================

    W2 -= learning_rate * dW2
    b2 -= learning_rate * db2

    W1 -= learning_rate * dW1
    b1 -= learning_rate * db1


    # Show progress every 10 epochs
    if (epoch + 1) % 10 == 0:
        predictions = np.argmax(A2, axis=1)
        accuracy = np.mean(predictions == y)

        print(
            f"Epoch {epoch + 1}, "
            f"Loss: {loss:.4f}, "
            f"Accuracy: {accuracy:.2f}"
        )

Epoch 10, Loss: 1.1012, Accuracy: 0.27
Epoch 20, Loss: 1.1009, Accuracy: 0.26
Epoch 30, Loss: 1.1006, Accuracy: 0.30
Epoch 40, Loss: 1.1002, Accuracy: 0.33
Epoch 50, Loss: 1.0999, Accuracy: 0.33
Epoch 60, Loss: 1.0996, Accuracy: 0.34
Epoch 70, Loss: 1.0993, Accuracy: 0.35
Epoch 80, Loss: 1.0990, Accuracy: 0.35
Epoch 90, Loss: 1.0987, Accuracy: 0.35
Epoch 100, Loss: 1.0984, Accuracy: 0.36


## 5. Check the 3D Softmax Jacobian


In [13]:
print("Softmax Jacobian shape:", J_softmax.shape)

print("\nJacobian for the first sample:")
print(J_softmax[0])

Softmax Jacobian shape: (100, 3, 3)

Jacobian for the first sample:
[[ 0.22153269 -0.1137103  -0.10782238]
 [-0.1137103   0.22542881 -0.1117185 ]
 [-0.10782238 -0.1117185   0.21954089]]


## 6. Check Final Predictions

The class with the highest probability is selected as the prediction.

In [14]:
predictions = np.argmax(A2, axis=1)

print("Actual labels:   ", y[:20])
print("Predicted labels:", predictions[:20])

final_accuracy = np.mean(predictions == y)

print("\nFinal Accuracy:", final_accuracy)

Actual labels:    [2 2 0 0 2 1 0 0 2 0 1 2 0 2 0 0 0 1 1 2]
Predicted labels: [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]

Final Accuracy: 0.36
